# L6 — Datadog Agent Observability

## What you'll learn

- How to find the sessions of your AnyCompany agents in Datadog Agent Observability
- How to read one conversation as a single session across the orchestrator and the specialist agents
- How to inspect the traces and the model calls in a session, including token usage
- How one customer conversation becomes one connected trace

## Overview

When you chose **Datadog** as the observability provider in the Partner Selection UI, the L3 notebooks set up the agents to send their traces straight to Datadog. Logs stay in CloudWatch.

This notebook doesn't set anything up. It runs one conversation and then walks through how it looks in Datadog:

- **Sessions.** The whole conversation, grouped as one session.
- **Traces.** Each request as a tree of steps across the orchestrator and the specialist agents.
- **Model calls.** The model, the tools and the token usage for each call.

## Architecture

![Observability layer: the agents emit OpenTelemetry spans. By default they go through managed ADOT to CloudWatch and CloudWatch Omni; with the Datadog option an OTLP exporter in the agent sends them to Datadog Agent Observability.](images/l6-observability-architecture.png)

This notebook follows the Datadog path at the bottom of the diagram: the agents send their traces to Datadog, and their logs stay in CloudWatch.

## Steps

0. Install dependencies
1. Load the configuration and set up the Datadog client
2. How Datadog Agent Observability fits
3. Generate traffic
4. Agent Observability: find your sessions
5. Open a session
6. APM traces and the span waterfall
7. LLM spans
8. One transaction = one session

## License details

Refer to LICENSE file in the main folder for license details.

## Prerequisites

- Datadog is your observability provider, and your Datadog keys and site are saved in the Partner Selection UI (Orchestration → Agent Observability).
- The L3 agents ([`2_specialist_agents.ipynb`](../l3-orchestration/2_specialist_agents.ipynb) and [`3_orchestrator_agent.ipynb`](../l3-orchestration/3_orchestrator_agent.ipynb)) were deployed with Datadog selected.
- There is trace data to view: you have chatted with the chatbot ([`2_chatbot_ui.ipynb`](../l5-application/2_chatbot_ui.ipynb)), or you run Step 3 below.
- You can sign in to your Datadog organization for the site you configured, for example `datadoghq.com`.


> **Screenshot** - showing the SSM parameters under `config/observability/` showing `provider=datadog`.

![screenshot: SSM config/observability parameters showing provider=datadog](images/datadog_ssm_provider.png)

## Step 0: Install Dependencies

`datadog-api-client` is used only for **read-only** Spans API queries (listing sessions, pulling a trace, reading LLM-span token usage). `boto3` reads SSM and invokes the Harness. The install is idempotent - safe to re-run.

In [ ]:
%pip install --quiet datadog-api-client==2.31.0 boto3

## Step 1: Load config & configure the Datadog client

Read the Datadog credentials L0 stored under `config/observability/` (`api_key`/`app_key` are SecureString, `site` is String), validate the `site` against the known Datadog hosts, and build a read-only `datadog_api_client` `Configuration`. The keys are **masked** on print - the raw values never appear in output.

In [ ]:
import os
import boto3
from datadog_api_client import Configuration

REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
SSM_PREFIX = "/anycompany/agentcore"
OBS_PREFIX = f"{SSM_PREFIX}/config/observability"

# Valid Datadog sites. The site becomes part of the API host, so only
# known values are accepted.
DATADOG_SITES = {
    "datadoghq.com",        # US1
    "us3.datadoghq.com",    # US3
    "us5.datadoghq.com",    # US5
    "datadoghq.eu",         # EU1
    "ap1.datadoghq.com",    # AP1
    "ddog-gov.com",         # US1-FED
}


def mask_secret(value):
    # Mask a secret for safe printing (first 4 + last 4 chars). Never print a raw key.
    if not value:
        return "<empty>"
    if len(value) <= 8:
        return "\u2026"
    return f"{value[:4]}\u2026{value[-4:]}"


ssm = boto3.client("ssm", region_name=REGION)


def _get(name, decrypt=False):
    try:
        return ssm.get_parameter(Name=name, WithDecryption=decrypt)["Parameter"]["Value"]
    except ssm.exceptions.ParameterNotFound as _e:
        raise RuntimeError(
            f"{name} not found in SSM. Re-run L0 (1_partner_selection) with Datadog "
            "selected for observability, then the L3 deploy notebooks."
        ) from _e


PROVIDER = _get(f"{OBS_PREFIX}/provider").strip().lower()
if PROVIDER != "datadog":
    raise RuntimeError(
        f"Observability provider is '{PROVIDER}', not 'datadog'. This is the Datadog "
        "notebook - if you chose CloudWatch in L0, run 1_cloudwatch_omni.ipynb instead."
    )

DD_API_KEY = _get(f"{OBS_PREFIX}/api_key", decrypt=True)   # SecureString
DD_APP_KEY = _get(f"{OBS_PREFIX}/app_key", decrypt=True)   # SecureString
DD_SITE = _get(f"{OBS_PREFIX}/site").strip()               # String

if DD_SITE not in DATADOG_SITES:
    raise ValueError(
        f"Unsupported Datadog site {DD_SITE!r}. Expected one of: "
        f"{', '.join(sorted(DATADOG_SITES))}."
    )

dd_config = Configuration()
dd_config.api_key["apiKeyAuth"] = DD_API_KEY
dd_config.api_key["appKeyAuth"] = DD_APP_KEY
dd_config.server_variables["site"] = DD_SITE

print("Datadog client configured (read-only).")
print(f"  site:    {DD_SITE}")
print(f"  api_key: {mask_secret(DD_API_KEY)}")
print(f"  app_key: {mask_secret(DD_APP_KEY)}")

## Step 2: How Datadog Agent Observability fits (OTEL-direct)

Datadog is a *different UI* on top of the same spans your agents already emit. Nothing in this notebook instruments anything - the wiring was done once in L3.

| | CloudWatch GenAI Observability | Datadog Agent Observability |
|---|---|---|
| **Where you work** | AWS Console / CloudWatch Omni | Datadog app at `https://app.<site>` |
| **What you query** | Logs Insights, X-Ray, Transaction Search | APM traces + Agent Observability sessions/LLM spans |
| **Agent observability** | Spans + metrics via AgentCore | Agent / LLM / Tool spans grouped into sessions |
| **Instrumentation** | Managed ADOT -> CloudWatch | OTEL-direct: app-owned OTLP exporter (specialists) + OTLP env vars (Harness). No in-container agent, no vendor SDK, no runtime guardrail layer. |
| **Logs** | CloudWatch | **Still CloudWatch** - only spans/traces go to Datadog |

The key resource attribute wired in L3, `dd_llmobs_enabled=true`, is what turns the GenAI-convention spans (emitted by strands-agents under `OTEL_SEMCONV_STABILITY_OPT_IN=gen_ai_latest_experimental`) into Datadog **Agent Observability** sessions. You do not set it here - you just view the result.

## Step 3: Generate traffic (one session, all four services)

Invoke the Harness several times in a **single session** with questions that force it to delegate to each specialist at least once:

- Turn 1 - an **order lookup** -> routes to `anycompany-order-agent`
- Turn 2 - a **refund request** -> routes to `anycompany-refund-agent`
- Turn 3 - a **product research** question -> routes to `anycompany-research-agent`

All turns reuse the same `runtimeSessionId`, so Datadog groups the orchestrator and all three specialists into **one session** (they each stamp `gen_ai.conversation.id` with this id). Note the printed `session_id` - you search for it in the UI in the next steps.

In [ ]:
import time
import uuid
from botocore.config import Config

ACCOUNT_ID = boto3.client("sts", region_name=REGION).get_caller_identity()["Account"]
HARNESS_ARN = _get(f"{SSM_PREFIX}/harness_arn")

# invoke_harness requires runtimeSessionId to be 33-100 chars. Pad a readable
# prefix with a uuid4 hex so the session is easy to spot in the Datadog UI.
session_id = f"dd-demo-{int(time.time())}-{uuid.uuid4().hex}"
actor_id = f"workshop-user-{ACCOUNT_ID[-4:]}"
print(f"Session ID: {session_id}")
print(f"Actor ID:   {actor_id}")

runtime = boto3.client("bedrock-agentcore", region_name=REGION, config=Config(read_timeout=300))

# Three turns that force delegation to ALL THREE specialists so every service
# reports spans under this one session:
QUESTIONS = [
    "Look up order ORD-10001 for customer CUST-789.",
    "I want to refund ORD-10001 because it arrived damaged - am I eligible?",
    "Can you research the best alternative wireless headphones I could buy instead?",
]

for turn, question in enumerate(QUESTIONS, start=1):
    print()
    print("=" * 72)
    print(f"Turn {turn}: {question}")
    print("=" * 72)
    response = runtime.invoke_harness(
        harnessArn=HARNESS_ARN,
        runtimeSessionId=session_id,
        actorId=actor_id,
        messages=[{"role": "user", "content": [{"text": question}]}],
    )
    for event in response["stream"]:
        if "contentBlockDelta" in event:
            delta = event["contentBlockDelta"].get("delta", {})
            if "text" in delta:
                print(delta["text"], end="", flush=True)
    print()

print()
print("-" * 72)
print(f"Done. In Datadog, filter by session_id = {session_id}")
print("Allow ~1-5 min for the OTLP spans to land before the next steps.")

### Read-only Spans API helper

The next steps optionally pull data inline. They all go through this one helper, which runs a **read-only** `SpansApi.list_spans` query - no monitors, dashboards, or SLOs are created anywhere in this notebook. Run this cell once before Steps 5-8.

In [ ]:
from datetime import datetime, timedelta, timezone
from datadog_api_client import ApiClient
from datadog_api_client.v2.api.spans_api import SpansApi
from datadog_api_client.v2.model.spans_list_request import SpansListRequest
from datadog_api_client.v2.model.spans_list_request_data import SpansListRequestData
from datadog_api_client.v2.model.spans_list_request_attributes import SpansListRequestAttributes
from datadog_api_client.v2.model.spans_list_request_page import SpansListRequestPage
from datadog_api_client.v2.model.spans_sort import SpansSort


def search_spans(query, minutes=30, limit=1000):
    # Read-only Spans API search. Returns a list of span attribute dicts.
    now = datetime.now(timezone.utc)
    with ApiClient(dd_config) as client:
        body = SpansListRequest(
            data=SpansListRequestData(
                type="search_request",
                attributes=SpansListRequestAttributes(
                    filter={
                        "from": (now - timedelta(minutes=minutes)).strftime("%Y-%m-%dT%H:%M:%SZ"),
                        "to": now.strftime("%Y-%m-%dT%H:%M:%SZ"),
                        "query": query,
                    },
                    sort=SpansSort.TIMESTAMP_DESCENDING,
                    page=SpansListRequestPage(limit=limit),
                ),
            )
        )
        data = SpansApi(client).list_spans(body=body).get("data", [])
    return [s.get("attributes", {}) for s in data]


def span_ms(a):
    # Duration in ms from the v2 Spans API start/end timestamps.
    st, en = a.get("start_timestamp"), a.get("end_timestamp")
    try:
        return (en - st).total_seconds() * 1000 if (st and en) else 0.0
    except Exception:
        return 0.0


SERVICES = [
    "anycompany-orchestrator",
    "anycompany-order-agent",
    "anycompany-refund-agent",
    "anycompany-research-agent",
]
print("Spans API helper ready. Services tracked:")
for s in SERVICES:
    print(f"  - {s}")

## Step 4: Agent Observability — find your sessions

Datadog **Agent Observability** goes beyond infrastructure monitoring - it captures the AI semantics of the system as three span kinds:

| Span Kind | What it represents | Example here |
|-----------|-------------------|--------------|
| **Agent** | One autonomous agent invocation | an `invoke_harness` turn of the orchestrator |
| **LLM** | A call to a language model | litellm -> Claude on Bedrock generating a response |
| **Tool** | An external tool the agent calls | `order-tools___check_order_details` via the Gateway |

In the UI go to **AI Observability > Traces** (also reachable at `https://app.<site>/llm/traces`). Switch the `ml_app` selector to `anycompany-orchestrator` to see the full orchestration, and pick the time range covering your Step 3 run. The optional cell below lists the recent sessions inline from the Spans API so you can confirm the data landed first.

In [ ]:
# OPTIONAL read-only pull: list the recent sessions (grouped by session_id) that
# the orchestrator emitted, newest first.
orch_spans = search_spans("@ml_app:anycompany-orchestrator", minutes=60)


def _session_id(a):
    # Datadog's session_id comes from gen_ai.conversation.id. Check the
    # locations the API exposes it under.
    custom = a.get("custom", {}) or {}
    gen_ai = (custom.get("gen_ai", {}) or {})
    conv = (gen_ai.get("conversation", {}) or {})
    return (
        conv.get("id")
        or custom.get("session_id")
        or (a.get("tags", {}) or {}).get("session_id")
        or "<ungrouped>"
    )


sessions = {}
for a in orch_spans:
    sid = _session_id(a)
    bucket = sessions.setdefault(sid, {"spans": 0, "services": set(), "last": None})
    bucket["spans"] += 1
    bucket["services"].add(a.get("service", "?"))
    ts = a.get("start_timestamp")
    if ts and (bucket["last"] is None or ts > bucket["last"]):
        bucket["last"] = ts

print(f"Recent orchestrator sessions: {len(sessions)}")
print("-" * 78)
print(f"{'session_id':<46} {'spans':>6} {'services':>9}")
print("-" * 78)
for sid, b in sorted(sessions.items(), key=lambda kv: kv[1]["last"] or datetime.min.replace(tzinfo=timezone.utc), reverse=True)[:10]:
    print(f"{sid[:45]:<46} {b['spans']:>6} {len(b['services']):>9}")

if not sessions:
    print("No orchestrator spans yet - run Step 3, wait ~1-5 min, then re-run this cell.")
print()
print(f"UI: https://app.{DD_SITE}/llm/traces?query=%40ml_app%3Aanycompany-orchestrator")

> **Screenshot:** the Agent Observability sessions list for the AnyCompany services.

![screenshot: Datadog Agent Observability sessions list for the AnyCompany services](images/datadog_sessions_list.png)

## Step 5: Open a session

Click into the session for your Step 3 conversation. The session view shows the end-to-end flow: the orchestrator Harness receiving each request, delegating to the specialists (order -> refund -> research), and the final responses. Each row is an Agent / LLM / Tool span; expanding the orchestrator's Agent span reveals the `execute_tool` delegations and the nested model calls.

Read it top-down: an **Agent** span (the orchestrator turn), its **LLM** spans (Claude deciding what to do), and the **Tool** spans where it calls the Gateway tools that reach the specialists. Each specialist then emits its own Agent/LLM spans, nested in the orchestrator's trace and the same session.

> **Screenshot:** a single Agent Observability session overview showing the orchestrator plus all three specialists.

![screenshot: a single Agent Observability session overview](images/datadog_session_overview.png)

## Step 6: APM traces & the span waterfall

Every service also emits a standard **APM trace**. From a session, open the linked APM trace, or use the optional cell below to find a representative orchestrator-rooted trace and render its span waterfall inline.

What the orchestrator trace contains:

```
POST /invocations                        <- root span (the Harness invocation)
  invoke_agent (Strands Agents)          <- the managed agent loop
    execute_event_loop_cycle   (xN)      <- one reasoning iteration each
      chat -> ConverseStream (Bedrock)   <- the orchestrator's own LLM call
      execute_tool                       <- delegation; the specialist's spans nest here
        invoke_agent (specialist)          <- joined via the propagated traceparent
      CreateEvent (AgentCore Memory)     <- short-term memory writes
```

**What you WILL see across the chain:** the orchestrator's `traceparent` propagates into each specialist (through the Gateway header allowlist and the `agent-invoker` Lambda), so the specialist spans adopt the orchestrator's **trace id** and join its trace. A delegation shows as an `execute_tool` span with the specialist's own Agent/LLM spans nested beneath it, and the **Service Map** draws the `anycompany-orchestrator -> anycompany-order-agent` / `-> refund` / `-> research` edges. You can still filter by `service:anycompany-order-agent` to isolate one specialist within the joined trace. (The managed `InvokeAgentRuntime` hop is not exported to Datadog, so nesting can show a one-span gap at that seam, but the shared trace id and the Service-Map edge hold.) The whole transaction is also grouped into **one session** (see Step 8).

In [ ]:
# OPTIONAL: find a representative orchestrator-rooted trace, then expand it by
# trace_id to pull every span across services.
FINDER_QUERY = 'service:anycompany-orchestrator -resource_name:"GET /ping"'
finder_spans = search_spans(FINDER_QUERY, minutes=30, limit=500)

traces = {}
for a in finder_spans:
    traces.setdefault(a.get("trace_id", ""), []).append(a)


def _is_real(spans):
    names = {s.get("operation_name", "") for s in spans} | {s.get("resource_name", "") for s in spans}
    return bool(names & {"invoke_agent", "execute_event_loop_cycle"}) or len(spans) >= 3


def _root_service(spans):
    ids = {s.get("span_id", "") for s in spans}
    for s in spans:
        if s.get("parent_id") in ("", "0", None) or s.get("parent_id") not in ids:
            return s.get("service")
    return None


candidates = {t: sp for t, sp in traces.items() if _is_real(sp)}


def _score(sp):
    return (_root_service(sp) == "anycompany-orchestrator", len({s.get("service") for s in sp}), len(sp))


TRACE_ID = max(candidates, key=lambda t: _score(candidates[t])) if candidates else None
print(f"Finder query: {FINDER_QUERY}")
print(f"Found {len(finder_spans)} orchestrator spans in {len(traces)} traces "
      f"({len(candidates)} pass the quality floor)")

if not TRACE_ID:
    print("\nNo representative orchestration trace in the window.")
    print("  -> Re-run Step 3, wait ~1-5 min, then re-run this cell.")
else:
    trace_spans = [a for a in search_spans(f"trace_id:{TRACE_ID}", minutes=30)
                   if a.get("resource_name") != "GET /ping"]
    if not trace_spans:
        trace_spans = traces[TRACE_ID]
    traces[TRACE_ID] = trace_spans
    svcs = sorted({s.get("service", "?") for s in trace_spans})
    print(f"Selected trace: {TRACE_ID}")
    print(f"  spans (all services): {len(trace_spans)}")
    print(f"  services in trace:    {', '.join(svcs)}")

### Render the span waterfall

Build the parent/child tree and print it as an indented waterfall, then break the wall-clock time down by **self-time** (a span's own duration minus its children's) so parent/child durations are not double-counted.

In [ ]:
if not TRACE_ID:
    print("No trace selected - run the previous cell first.")
else:
    tspans = sorted(traces[TRACE_ID],
                    key=lambda s: s.get("start_timestamp").timestamp() if s.get("start_timestamp") else 0.0)
    span_map = {s.get("span_id", ""): s for s in tspans}
    children = {}
    roots = []
    for s in tspans:
        pid = s.get("parent_id", "")
        if pid and pid in span_map:
            children.setdefault(pid, []).append(s.get("span_id", ""))
        else:
            roots.append(s.get("span_id", ""))

    display = []

    def walk(sid, depth=0):
        s = span_map.get(sid, {})
        display.append({
            "name": s.get("resource_name", s.get("operation_name", "?")),
            "service": s.get("service", "?"),
            "depth": depth,
            "ms": span_ms(s),
            "status": s.get("status", "ok"),
        })
        for c in children.get(sid, []):
            walk(c, depth + 1)

    for r in roots:
        walk(r)
    if not display:
        display = [{"name": s.get("resource_name", "?"), "service": s.get("service", "?"),
                    "depth": 0, "ms": span_ms(s), "status": s.get("status", "ok")} for s in tspans]

    print(f"Trace: {TRACE_ID}")
    print("=" * 90)
    print(f'{"Span":<42} {"Service":<22} {"Duration":>9} {"Status":>6}')
    print("-" * 90)
    for i in display:
        indent = "  " * i["depth"]
        pfx = "|-- " if i["depth"] > 0 else ""
        n = f"{indent}{pfx}{i['name']}"[:41]
        st = "ERR" if i["status"] == "error" else "ok"
        print(f"{n:<42} {i['service']:<22} {i['ms']:>7.1f}ms {st:>6}")
    print("=" * 90)

    # Self-time breakdown.
    child_ms = {}
    for s in tspans:
        pid = s.get("parent_id", "")
        if pid in span_map:
            child_ms[pid] = child_ms.get(pid, 0.0) + span_ms(s)
    rows = []
    for s in tspans:
        self_ms = max(span_ms(s) - child_ms.get(s.get("span_id", ""), 0.0), 0.0)
        rows.append((s.get("resource_name", s.get("operation_name", "?")), s.get("service", "?"), self_ms))
    total = sum(r[2] for r in rows)
    print()
    print("Self-time by span (top 10) - where wall-clock is actually spent:")
    print("-" * 72)
    for name, svc, ms in sorted(rows, key=lambda r: -r[2])[:10]:
        pct = f"{ms / total * 100:4.0f}%" if total else "   -"
        print(f"  {name[:34]:<34} {svc[:20]:<20} {ms:>8.0f}ms {pct}")
    print(f'  {"TOTAL (self-time)":<34} {"":<20} {total:>8.0f}ms')
    print()
    print(f"UI flame graph: https://app.{DD_SITE}/apm/trace/{TRACE_ID}")

> **Screenshot:** the APM trace flame graph for a session.

![screenshot: APM trace flame graph for a session](images/datadog_apm_trace.png)

## Step 7: LLM spans (model, tokens, prompt/tools)

Within a trace, open an **LLM span** to see the model call detail: the input messages, tool invocations, the model output, and token usage. This is where the GenAI 1.37+ semantic-convention attributes emitted by strands-agents surface (`gen_ai.request.model`, `gen_ai.usage.*`). The optional cell pulls the per-call model/token/latency detail inline from the `gen_ai` spans.

In [ ]:
# OPTIONAL read-only pull: per-LLM-call model, token usage, and latency across
# all four services, straight from the gen_ai spans.
def _usage(a):
    return (a.get("custom", {}).get("gen_ai", {}) or {}).get("usage", {}) or {}


llm_rows = [a for a in search_spans("service:anycompany-*", minutes=30)
            if _usage(a).get("input_tokens") is not None]
if not llm_rows:
    llm_rows = [a for a in search_spans("service:anycompany-*", minutes=360)
                if _usage(a).get("input_tokens") is not None]

print(f"LLM calls found: {len(llm_rows)}\n")
print(f'{"Service":<28} {"Model":<30} {"in tok":>8} {"out tok":>8} {"ms":>7}')
print("-" * 86)
tot_in = tot_out = 0
for a in llm_rows[:25]:
    ga = a.get("custom", {}).get("gen_ai", {}) or {}
    u = ga.get("usage", {}) or {}
    model = (ga.get("request", {}) or {}).get("model", "?")
    i, o = int(u.get("input_tokens", 0) or 0), int(u.get("output_tokens", 0) or 0)
    tot_in += i
    tot_out += o
    print(f'{a.get("service", "?")[:27]:<28} {str(model)[:29]:<30} {i:>8,} {o:>8,} {span_ms(a):>7.0f}')
print("-" * 86)
print(f'{"TOTAL":<28} {"":<30} {tot_in:>8,} {tot_out:>8,}')
if not llm_rows:
    print("\nNo gen_ai LLM spans found - run Step 3 (drive traffic), wait, then re-run.")
print()
print(f"UI: https://app.{DD_SITE}/llm/traces?query=%40ml_app%3Aanycompany-orchestrator")

> **Screenshot:** an expanded LLM span showing prompt, tools, and token usage.

![screenshot: an expanded LLM span showing prompt, tools, and token usage](images/datadog_llm_span.png)

## Step 8: One transaction = one session

Delegated turns **join into one trace**: the orchestrator's `traceparent` propagates into each specialist, so their spans share the orchestrator's trace id. On top of that, Datadog groups the **entire** conversation - every turn, whether it delegates or not - into one session via the shared `session_id`, set from the AgentCore runtime session id through the `gen_ai.conversation.id` attribute.

The optional cell confirms that the session from Step 3 contains the orchestrator **and all three specialists**.

In [ ]:
# OPTIONAL: confirm the Step 3 session spans all four services.
if "session_id" not in dir():
    print("Run Step 3 first - session_id is not defined.")
else:
    sess_spans = search_spans(f'@gen_ai.conversation.id:"{session_id}"', minutes=60)
    if not sess_spans:
        # Fallback: some span sets expose the id as a plain tag rather than an attribute.
        sess_spans = [a for a in search_spans("service:anycompany-*", minutes=60)
                      if _session_id(a) == session_id]
    svcs = sorted({a.get("service", "?") for a in sess_spans})
    print(f"session_id: {session_id}")
    print(f"spans in session: {len(sess_spans)}")
    print(f"services in session: {', '.join(svcs) if svcs else '(none yet)'}")
    expected = set(SERVICES)
    missing = expected - set(svcs)
    if sess_spans and not missing:
        print("\nAll four services present under one session.")
    elif sess_spans:
        print(f"\nNot all services visible yet (missing: {', '.join(sorted(missing))}).")
        print("OTLP conversion can lag 3-5 min; wait and re-run, or confirm the turn")
        print("that routes to the missing specialist actually ran in Step 3.")
    else:
        print("\nNo spans for this session yet - wait ~1-5 min after Step 3 and re-run.")

> **Screenshot:** one session grouping orchestrator + order + refund + research spans by shared `session_id`.

![screenshot: one session grouping orchestrator + order + refund + research spans by shared session_id](images/datadog_one_session.png)

> If the spans appear as separate sessions instead, re-check that the chatbot passes one `runtimeSessionId` per conversation, so all spans share the same session id.

## Summary

This notebook ran one conversation through the orchestrator and the specialist agents and followed it in Datadog: as one session, as connected traces and as model calls with their token usage. Traces went to Datadog, and logs stayed in CloudWatch.

**Next:** [`1_agent_evaluations.ipynb`](../l7-agentic-operations/1_agent_evaluations.ipynb). Score how well the agents handle conversations.

---
## Cleanup

Datadog is **SaaS - there is nothing to tear down**. This notebook created no resources: it only ran read-only Spans API queries and drove traffic through the already-deployed agents. Traces expire automatically per your Datadog retention policy.

**Do not delete** the shared `config/observability/*` SSM parameters here - they are owned by the L0 selector and read by the L3 deploys. To stop the agents from emitting spans to Datadog, re-run L0 and pick CloudWatch, then redeploy the L3 agents.

In [ ]:
# === CLEANUP ===
# Datadog is SaaS, so there is nothing to delete here. This notebook provisioned
# no resources - only read-only Spans API queries plus traffic through the
# already-deployed agents. Traces in Datadog expire per your retention policy.
#
# Do NOT delete /anycompany/agentcore/config/observability/* here - those params
# are owned by L0 and read by the L3 deploy notebooks.
print("Nothing to clean up - Datadog is SaaS and this notebook created no resources.")